In [1]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import xarray as xr
from glob import glob
import os
from netCDF4 import Dataset
import pandas as pd
from datetime import datetime, date, timedelta
from pathlib import Path
import scipy
import scipy.ndimage
from mpl_toolkits.axes_grid1 import ImageGrid
import math
import cc3d

from mpl_toolkits.axes_grid1 import make_axes_locatable

#constants
t = 6 # time index
target_idx = 1
z_min = 0
z_max = 200

periodic = True
dilated_ql = True

#Input
input_dir = Path("/mnt/stor-pool-01/projects/heus/ShellAnalysis/full-area/")
source_input_dir = Path("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/")

In [ ]:
#load datasets
#ds_ql_mask = xr.open_dataset(input_dir / "ql_mask.nc", decode_times=False)
ds_cloud_mask = xr.open_dataset(input_dir / "cloud_mask.nc", decode_times=False)
ds_w_mask = xr.open_dataset(input_dir / "w_mask.nc", decode_times=False)
ds_shell_mask = xr.open_dataset(input_dir / "shell_mask.nc", decode_times=False)
ds_shell_labels = xr.open_dataset(input_dir / "shell_labels.nc", decode_times=False)
ds_cloud_labels = xr.open_dataset(input_dir / "cloud_labels.nc", decode_times=False)

ds_ql = xr.open_dataset(source_input_dir / "ql.nc", decode_times=False,chunks={'time': 1})
ds_w = xr.open_dataset(source_input_dir / "w.nc", decode_times=False,chunks={'time': 1})
ds_w = ds_w.rename({'zh':'z'}).interp(z=ds_ql.z)

In [ ]:
shell_labels_slice_z_range = ds_shell_labels.isel(time=t).sel(z=slice(z_min, z_max))
low_shell_ids = np.unique(shell_labels_slice_z_range.shell_labels.where(shell_labels_slice_z_range.shell_labels > 0).values)

shell_mask_slice = ds_shell_mask.isel(time=t)
cloud_mask_slice = ds_cloud_mask.isel(time=t)
shell_labels_slice = ds_shell_labels.isel(time=t)
cloud_labels_slice = ds_cloud_labels.isel(time=t)

In [ ]:
depth_3d_full_shell = shell_mask_slice["shell_mask"].where(shell_mask_slice["shell_mask"] != 0).values
depth_3d_full_cloud = cloud_mask_slice["cloud_mask"].where(cloud_mask_slice["cloud_mask"] != 0).values

#extract labels
shell_labels_3d_slice = shell_labels_slice.shell_labels.values[:, :, :]
cloud_labels_3d_slice = cloud_labels_slice.cloud_labels.values[:, :, :]


#apply labels
target_id = low_shell_ids[target_idx]
depth_3d_full_shell = np.where(shell_labels_3d_slice == target_id, depth_3d_full_shell, np.nan)
depth_3d_full_cloud = np.where(cloud_labels_3d_slice == target_id, depth_3d_full_cloud, np.nan)

is_shell = ~np.isnan(depth_3d_full_shell)
is_cloud = ~np.isnan(depth_3d_full_cloud)

if(dilated_ql):
    #assemble dilated version
    expansion = np.zeros((3,3,3), dtype=bool)
    expansion[1, 1, :] = True  # X axis
    expansion[1, :, 1] = True  # Y axis
    expansion[:, 1, 1] = True  # Z axis

    dilated_is_cloud = np.pad(is_cloud, ((1, 1), (0, 0), (0, 0)), mode='constant', constant_values=0)
    dilated_is_cloud = np.pad(dilated_is_cloud, ((0, 0), (1, 1), (1, 1)), mode='wrap')
    dilated_is_cloud = scipy.ndimage.binary_dilation(dilated_is_cloud, structure=expansion, iterations=1)
    dilated_is_cloud = dilated_is_cloud[1:-1, 1:-1, 1:-1]
    merged_np = is_shell | dilated_is_cloud
else:
    merged_np = is_shell | is_cloud



In [5]:
#Validate connectivity

# 1. Get the list of ALL true coordinate indices in the mask
true_z, true_y, true_x = np.where(merged_np)
total_valid_voxels = len(true_z)

if total_valid_voxels == 0:
    print("❌ The mask is completely empty.")
else:
    # Convert true points into a fast-lookup lookup set of tuples (z, y, x)
    all_points_set = set(zip(true_z, true_y, true_x))

    # 2. Setup BFS tracking structures
    # Start at the very first valid voxel found
    start_voxel = (true_z[0], true_y[0], true_x[0])
    print(f"First node: (z={true_z[0]}, y={true_y[0]}, x={true_x[0]} (indexes))")

    queue = [start_voxel]
    visited = set([start_voxel])

    # Get grid dimensions for cyclic boundary wrapping on Y and X
    nz, ny, nx = merged_np.shape

    print(f"Starting manual verification for {total_valid_voxels} voxels...")

    # 3. Fast Queue execution loop
    head = 0
    while head < len(queue):
        curr_z, curr_y, curr_x = queue[head]
        if head != 0:
            print(f"Current node: (z={curr_z}, y={curr_y}, x={curr_x} (indexes))")
        head += 1

        # Define 6 structural neighbors with periodic wrapping on Y and X
        if periodic:
            neighbors = [
                (curr_z + 1, curr_y, curr_x),  # Up
                (curr_z - 1, curr_y, curr_x),  # Down
                
                (curr_z, (curr_y + 1) % ny, curr_x),  # Forward (Periodic)
                (curr_z, (curr_y - 1) % ny, curr_x),  # Backward (Periodic)
                (curr_z, curr_y, (curr_x + 1) % nx),  # Right (Periodic)
                (curr_z, curr_y, (curr_x - 1) % nx),  # Left (Periodic)
            ]
        else:
            neighbors = [
                (curr_z + 1, curr_y, curr_x),  # Up
                (curr_z - 1, curr_y, curr_x),  # Down

                #Checking non-periodic
                (curr_z, curr_y + 1, curr_x),  # Forward
                (curr_z, curr_y - 1, curr_x),  # Backward
                (curr_z, curr_y, curr_x + 1),  # Right
                (curr_z, curr_y, curr_x - 1),  # Left
            ]

        for n_voxel in neighbors:
            # Drop check if it falls outside vertical boundaries
            print(f"Checking at z={n_voxel[0]}, y={n_voxel[1]}, x={n_voxel[2]}", end="")
            if n_voxel[0] < 0 or n_voxel[0] >= nz:
                print("-> EXCLUDED (Result: outside of z bounds)")
                continue

            if(periodic):
                if n_voxel[1] < 0 or n_voxel[1] >= ny:
                    print("-> EXCLUDED (Result: outside of y bounds)")
                    continue
                if n_voxel[2] < 0 or n_voxel[2] >= nx:
                    print("-> EXCLUDED (Result: outside of x bounds)")
                    continue
                
            # If the neighbor is part of the cloud and we haven't visited it yet
            if n_voxel in all_points_set and n_voxel not in visited:
                print("-> ADDED TO QUEUE AND VISITED LISTS (VALID)")
                visited.add(n_voxel)
                queue.append(n_voxel)
            elif n_voxel not in all_points_set:
                print("-> EXCLUDED (Result: not in shape)")
            else:
                print("-> EXCLUDED (Already visited)")
    
    # 4. Final Connection Audit
    print("-" * 40)
    print(f"Total voxels in cloud structure: {total_valid_voxels}")
    print(f"Voxels reached from start point: {len(visited)}")
    
    if len(visited) == total_valid_voxels:
        print("✅ Success! The structure is 100% manually verified as fully connected.")
    else:
        print(f"❌ Fragmented! Contains {total_valid_voxels - len(visited)} isolated voxel components.")

First node: (z=2, y=0, x=619 (indexes))
Starting manual verification for 1012 voxels...
Checking at z=3, y=0, x=619-> ADDED TO QUEUE AND VISITED LISTS (VALID)
Checking at z=1, y=0, x=619-> EXCLUDED (Result: not in shape)
Checking at z=2, y=1, x=619-> EXCLUDED (Result: not in shape)
Checking at z=2, y=1023, x=619-> EXCLUDED (Result: not in shape)
Checking at z=2, y=0, x=620-> EXCLUDED (Result: not in shape)
Checking at z=2, y=0, x=618-> EXCLUDED (Result: not in shape)
Current node: (z=3, y=0, x=619 (indexes))
Checking at z=4, y=0, x=619-> ADDED TO QUEUE AND VISITED LISTS (VALID)
Checking at z=2, y=0, x=619-> EXCLUDED (Already visited)
Checking at z=3, y=1, x=619-> EXCLUDED (Result: not in shape)
Checking at z=3, y=1023, x=619-> EXCLUDED (Result: not in shape)
Checking at z=3, y=0, x=620-> EXCLUDED (Result: not in shape)
Checking at z=3, y=0, x=618-> EXCLUDED (Result: not in shape)
Current node: (z=4, y=0, x=619 (indexes))
Checking at z=5, y=0, x=619-> ADDED TO QUEUE AND VISITED LISTS (V

In [6]:
"""
#Setting up lists for connectivity
nz, ny, nx = ds_ql.ql.shape[1:]
z_grid, y_grid, x_grid = np.indices((nz, ny, nx))

def evaluate(np_binary, z, y, x, nz, ny, nx):
    if(z < 0 or y < 0 or x < 0): #below 0
        return False
    if(~(z < len(nz) & y < len(ny) & x < len(nx))):
        return False
    return np_binary[z,y,x]

list_idx = 0
neighbor_sets = []
for i in x_grid:
    for j in y_grid:
        for k in z_grid:
            if evaluate(merged_np, k, j, i, z_grid, y_grid, x_grid):
                neighbor_sets.append([]) #begin list for this point
                neighbor_sets[list_idx].append([k,j,i]) # add this point itself

                #up
                if evaluate(merged_np, k+1, j, i, z_grid, y_grid, x_grid):
                    neighbor_sets[list_idx].append([k+1,j,i])
                #down
                if evaluate(merged_np, k-1, j, i, z_grid, y_grid, x_grid):
                    neighbor_sets[list_idx].append([k-1,j,i])
                #left
                if evaluate(merged_np, k, j, i-1, z_grid, y_grid, x_grid):
                    neighbor_sets[list_idx].append([k,j,i-1])
                #right
                if evaluate(merged_np, k, j, i+1, z_grid, y_grid, x_grid):
                    neighbor_sets[list_idx].append([k,j,i+1])
                #forward
                if evaluate(merged_np, k, j+1, i, z_grid, y_grid, x_grid):
                    neighbor_sets[list_idx].append([k,j+1,i])
                #backward
                if evaluate(merged_np, k, j-1, i, z_grid, y_grid, x_grid):
                    neighbor_sets[list_idx].append([k,j-1,i])

                list_idx += 1

"""

'\n#Setting up lists for connectivity\nnz, ny, nx = ds_ql.ql.shape[1:]\nz_grid, y_grid, x_grid = np.indices((nz, ny, nx))\n\ndef evaluate(np_binary, z, y, x, nz, ny, nx):\n    if(z < 0 or y < 0 or x < 0): #below 0\n        return False\n    if(~(z < len(nz) & y < len(ny) & x < len(nx))):\n        return False\n    return np_binary[z,y,x]\n\nlist_idx = 0\nneighbor_sets = []\nfor i in x_grid:\n    for j in y_grid:\n        for k in z_grid:\n            if evaluate(merged_np, k, j, i, z_grid, y_grid, x_grid):\n                neighbor_sets.append([]) #begin list for this point\n                neighbor_sets[list_idx].append([k,j,i]) # add this point itself\n\n                #up\n                if evaluate(merged_np, k+1, j, i, z_grid, y_grid, x_grid):\n                    neighbor_sets[list_idx].append([k+1,j,i])\n                #down\n                if evaluate(merged_np, k-1, j, i, z_grid, y_grid, x_grid):\n                    neighbor_sets[list_idx].append([k-1,j,i])\n             

In [7]:
"""
start_idx = 0

master_set = neighbor_sets[start_idx].copy()
neighbor_sets.remove(neighbor_sets[start_idx])

iterations = 1
no_convergence_iteration_count = len(neighbor_sets + 1)

def check_pair(item1, item2):
    if (item1[0] == item2[0] & item1[1] == item2[1] & item1[2] == item2[2]):
        return True
    return False

def add_unique_items(original_set, new_set):
    for o_idx in range(len(original_set)):
        for n_idx in range(len(new_set)):
            if check_pair(original_set[o_idx], new_set[n_idx]):
                original_set.append(new_set[n_idx].copy())

    return original_set
            
pair_found = False
while(iterations < no_convergence_iteration_count):
    print(f"Iteration {iterations} / {no_convergence_iteration_count - 1}")
    pair_found = False
    for idx in range(len(neighbor_sets)):
        checked_set = neighbor_sets[idx]
        for m_idx in range(len(master_set)):
            m_item = master_set[m_idx]
            for c_idx in range(len(checked_set)):
                c_item = checked_set[c_idx]

                #check if they share an item
                pair_found = check_pair(m_item, c_item)

                if pair_found:
                    master_set = add_unique_items(master_set, checked_set)
                    neighbor_sets.remove(neighbor_sets[c_idx])
                    break

            if pair_found:
                break

        if pair_found:
            break
    iterations += 1

                
if(iterations >= no_convergence_iteration_count):
    print("Not connected")
else:
    print(f"Connected in {iterations} iterations")
    
"""

'\nstart_idx = 0\n\nmaster_set = neighbor_sets[start_idx].copy()\nneighbor_sets.remove(neighbor_sets[start_idx])\n\niterations = 1\nno_convergence_iteration_count = len(neighbor_sets + 1)\n\ndef check_pair(item1, item2):\n    if (item1[0] == item2[0] & item1[1] == item2[1] & item1[2] == item2[2]):\n        return True\n    return False\n\ndef add_unique_items(original_set, new_set):\n    for o_idx in range(len(original_set)):\n        for n_idx in range(len(new_set)):\n            if check_pair(original_set[o_idx], new_set[n_idx]):\n                original_set.append(new_set[n_idx].copy())\n\n    return original_set\n            \npair_found = False\nwhile(iterations < no_convergence_iteration_count):\n    print(f"Iteration {iterations} / {no_convergence_iteration_count - 1}")\n    pair_found = False\n    for idx in range(len(neighbor_sets)):\n        checked_set = neighbor_sets[idx]\n        for m_idx in range(len(master_set)):\n            m_item = master_set[m_idx]\n            for